### 2 - Teste Cálculos Indicadores

#### 2.1 Importação dos dados tratados e verificação dos dados:

In [37]:
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", None)

df_dados_tratados = pd.read_parquet("../dados_tratados/restinga_dados_tratados.parquet")

print("Dimensão original df_dados_tratados:", df_dados_tratados.shape)

Dimensão original df_dados_tratados: (10445, 18)


In [38]:
df_dados_tratados.head()

,Ano,Categoria da Situação,Cor / Raça,Código da Matricula,Código do Ciclo Matricula,Data de Fim Previsto do Ciclo,Data de Inicio do Ciclo,Data de Ocorrencia da Matricula,Eixo Tecnológico,Faixa Etária,Mês De Ocorrência da Situação,Nome de Curso,Renda Familiar,Sexo,Situação de Matrícula,Subeixo Tecnológico,Tipo de Curso,Turno
0,2017,Em curso,Não declarada,44681986,1207788,2014-12-22,2012-03-05,2012-03-01,Informação e Comunicação,35 a 39 anos,2018-03-05,Análise e Desenvolvimento de Sistemas,Não declarada,F,Retido,Informática,Superior-Tecnologia,Matutino
1,2017,Em curso,Branca,66262145,2007734,2019-12-20,2016-02-22,2016-02-01,Controle e Processos Industriais,15 a 19 anos,2016-02-01,Técnico em Eletrônica,"0,5<RFP<=1",F,Em fluxo,Elétrica,Técnico-Integrado,Matutino
2,2017,Em curso,Branca,66262155,2007734,2019-12-20,2016-02-22,2016-02-01,Controle e Processos Industriais,15 a 19 anos,2016-02-01,Técnico em Eletrônica,"0,5<RFP<=1",F,Em fluxo,Elétrica,Técnico-Integrado,Matutino
3,2017,Em curso,Branca,66262151,2007734,2019-12-20,2016-02-22,2016-02-01,Controle e Processos Industriais,15 a 19 anos,2016-02-01,Técnico em Eletrônica,"0<RFP<=0,5",F,Em fluxo,Elétrica,Técnico-Integrado,Matutino
4,2017,Evadidos,Branca,66262139,2007734,2019-12-20,2016-02-22,2016-02-01,Controle e Processos Industriais,15 a 19 anos,2017-05-01,Técnico em Eletrônica,"0<RFP<=0,5",F,Transf. externa,Elétrica,Técnico-Integrado,Matutino


In [39]:
df_dados_tratados.info() 

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10445 entries, 0 to 10444
Data columns (total 18 columns):
 #   Column                           Non-Null Count  Dtype         
---  ------                           --------------  -----         
 0   Ano                              10445 non-null  int64         
 1   Categoria da Situação            10445 non-null  object        
 2   Cor / Raça                       10445 non-null  object        
 3   Código da Matricula              10445 non-null  int64         
 4   Código do Ciclo Matricula        10445 non-null  int64         
 5   Data de Fim Previsto do Ciclo    10445 non-null  datetime64[ns]
 6   Data de Inicio do Ciclo          10445 non-null  datetime64[ns]
 7   Data de Ocorrencia da Matricula  10445 non-null  datetime64[ns]
 8   Eixo Tecnológico                 10445 non-null  object        
 9   Faixa Etária                     10445 non-null  object        
 10  Mês De Ocorrência da Situação    10445 non-null  datetime6

#### 2.2 Cálculos dos Indicadores de Permanência e Êxito:

In [40]:
# função para calcular os indicadores:

df_dados_tratados["Concluinte_Previsto"] = (
    df_dados_tratados["Data de Fim Previsto do Ciclo"].dt.year == df_dados_tratados["Ano"]
)

def calcular_indicadores(df, agrupamento):
    df_indicadores = (
        df.groupby(agrupamento)
          .agg(
            matr_atendidas = ("Código da Matricula", "nunique"),
            conc_no_prazo = ("Situação de Matrícula", lambda x: (x == "Concluída no prazo").sum()),
            conc_com_atraso = ("Situação de Matrícula", lambda x: (x == "Concluída com atraso").sum()),
            abandono = ("Situação de Matrícula", lambda x: (x == "Abandono").sum()),
            desligados = ("Situação de Matrícula", lambda x: (x == "Desligada").sum()),
            transf_ext = ("Situação de Matrícula", lambda x: (x == "Transf. externa").sum()),
            transf_int = ("Situação de Matrícula", lambda x: (x == "Transf. interna").sum()),
            integralizadas = ("Situação de Matrícula", lambda x: (x == "Integralizada").sum()),
            conc_previstos=("Concluinte_Previsto", "sum"),
            MREG = ("Situação de Matrícula", lambda x: (x == "Em fluxo").sum()), # MREG = matrículas ativas regulares
            MRET = ("Situação de Matrícula", lambda x: (x == "Retido").sum()), #MRET = matrículas ativas retidas
          )
          .reset_index()
    )
    
    # Cálculo da taxa de conclusão (TC)
    df_indicadores["TC"] = ( 
        ((df_indicadores["conc_no_prazo"] + df_indicadores["conc_com_atraso"]) / 
        df_indicadores["matr_atendidas"]) * 100
    )
    
    # Cálculo da taxa de evasão (TE)
    df_indicadores["TE"] = (
        ((df_indicadores["abandono"] + df_indicadores["desligados"] + df_indicadores["transf_ext"]) / 
        df_indicadores["matr_atendidas"]) * 100
    )
    
    # Cálculo da taxa de retenção (TR)
    df_indicadores["TR"] = (
        (df_indicadores["MRET"] / df_indicadores["matr_atendidas"]) * 100
    )
    
    # Cálculo do Índice de Eficiência (IEf)
    df_indicadores["IEf"] = (
        (df_indicadores["conc_no_prazo"] / 
            (df_indicadores["conc_no_prazo"] + 
            df_indicadores["conc_com_atraso"] +
            df_indicadores["abandono"] + df_indicadores["desligados"] +
            df_indicadores["transf_int"]+
             df_indicadores["transf_ext"])) * 100
    )
    
    # Cálculo da taxa de matrícula continuada regular:
    df_indicadores["t_matr_cont_reg"] = (
        df_indicadores["MREG"]/df_indicadores["matr_atendidas"] * 100
    )
    
    # Cálculo da Taxa de Permanência e Êxito: 
    df_indicadores["TPE"] = (
        df_indicadores["TC"] + df_indicadores["t_matr_cont_reg"]
    )
    
    # Cálculo da taxa de efetividade acadêmica:
    df_indicadores["TEFAcad"] = (
        df_indicadores["conc_no_prazo"]/df_indicadores["conc_previstos"]
    )
    
    # Após o cálculo dos indicadores, preencher os valores ausentes com zero
    df_indicadores = df_indicadores.fillna(0) 

    return df_indicadores.round(2)


In [41]:
df_indicadores_ano = calcular_indicadores(df_dados_tratados, ["Ano"])
df_indicadores_ano.head(20)

,Ano,matr_atendidas,conc_no_prazo,conc_com_atraso,abandono,desligados,transf_ext,transf_int,integralizadas,conc_previstos,MREG,MRET,TC,TE,TR,IEf,t_matr_cont_reg,TPE,TEFAcad
0,2017,811,48,20,70,45,13,2,2,105,569,42,8.38,15.78,5.18,24.24,70.16,78.55,0.46
1,2018,1026,67,13,40,46,15,1,0,155,765,79,7.80,9.84,7.70,36.81,74.56,82.36,0.43
2,2019,1285,61,17,103,33,6,0,1,227,991,73,6.07,11.05,5.68,27.73,77.12,83.19,0.27
3,2020,1223,0,2,78,10,4,3,0,285,960,166,0.16,7.52,13.57,0.00,78.50,78.66,0.00
4,2021,1122,0,28,27,18,19,0,0,319,762,268,2.50,5.70,23.89,0.00,67.91,70.41,0.00
5,2022,1497,50,62,83,36,20,0,0,317,728,518,7.48,9.29,34.60,19.92,48.63,56.11,0.16
6,2023,1669,97,66,47,23,18,0,0,203,885,533,9.77,5.27,31.94,38.65,53.03,62.79,0.48
7,2024,1812,56,51,1,33,8,0,2,212,1024,637,5.91,2.32,35.15,37.58,56.51,62.42,0.26


In [42]:
df_indicadores_ano.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8 entries, 0 to 7
Data columns (total 19 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   Ano              8 non-null      int64  
 1   matr_atendidas   8 non-null      int64  
 2   conc_no_prazo    8 non-null      int64  
 3   conc_com_atraso  8 non-null      int64  
 4   abandono         8 non-null      int64  
 5   desligados       8 non-null      int64  
 6   transf_ext       8 non-null      int64  
 7   transf_int       8 non-null      int64  
 8   integralizadas   8 non-null      int64  
 9   conc_previstos   8 non-null      int64  
 10  MREG             8 non-null      int64  
 11  MRET             8 non-null      int64  
 12  TC               8 non-null      float64
 13  TE               8 non-null      float64
 14  TR               8 non-null      float64
 15  IEf              8 non-null      float64
 16  t_matr_cont_reg  8 non-null      float64
 17  TPE              8 n

In [43]:
df_indicadores_curso_ano = calcular_indicadores(df_dados_tratados, ["Ano", "Nome de Curso"])
df_indicadores_curso_ano.head(10)

,Ano,Nome de Curso,matr_atendidas,conc_no_prazo,conc_com_atraso,abandono,desligados,transf_ext,transf_int,integralizadas,conc_previstos,MREG,MRET,TC,TE,TR,IEf,t_matr_cont_reg,TPE,TEFAcad
0,2017,Análise e Desenvolvimento de Sistemas,247,5,12,25,8,2,1,0,27,163,31,6.88,14.17,12.55,9.43,65.99,72.87,0.19
1,2017,Eletrônica Industrial,84,0,0,19,6,0,0,0,11,59,0,0.00,29.76,0.00,0.00,70.24,70.24,0.00
2,2017,Gestão Desportiva e de Lazer,112,8,4,13,5,0,0,2,20,74,6,10.71,16.07,5.36,26.67,66.07,76.79,0.40
3,2017,Letras - Português e Espanhol,30,0,0,0,5,0,0,0,0,25,0,0.00,16.67,0.00,0.00,83.33,83.33,0.00
4,2017,Técnico em Comércio,33,0,0,0,0,0,0,0,0,33,0,0.00,0.00,0.00,0.00,100.00,100.00,0.00
5,2017,Técnico em Eletrônica,115,11,3,4,10,6,1,0,23,76,4,12.17,17.39,3.48,31.43,66.09,78.26,0.48
6,2017,Técnico em Guia de Turismo,87,24,1,9,10,0,0,0,24,42,1,28.74,21.84,1.15,54.55,48.28,77.01,1.00
7,2017,Técnico em Informática,48,0,0,0,0,2,0,0,0,46,0,0.00,4.17,0.00,0.00,95.83,95.83,0.00
8,2017,Técnico em Lazer,55,0,0,0,1,3,0,0,0,51,0,0.00,7.27,0.00,0.00,92.73,92.73,0.00
9,2018,Análise e Desenvolvimento de Sistemas,267,10,4,15,11,0,0,0,46,173,54,5.24,9.74,20.22,25.00,64.79,70.04,0.22


In [44]:
df_indicadores_curso_ano.to_parquet("teste_indicadores_curso_ano.parquet", index=False)